我要尝试一下 自己使用 FFN 来求解 H2分子的系统基态

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import optax
from tqdm import tqdm
from functools import partial

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz (Flax Linen 版本)
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16
    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex128)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(1, param_dtype=jnp.complex128)(x)
        return x.squeeze(-1)


# ==============================================================================
# 4. 初始化模型、采样器、优化器
# ==============================================================================
model = SingleStateAnsatz(hidden_dim=12)

dummy_input = hi.random_state(jax.random.key(0), size=1)
params = model.init(jax.random.key(21), dummy_input)

# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)
sampler_state = sampler.init_state(model, params, seed=1)

optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(params)



/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [8]:
haa = ha.to_dense()


In [10]:
haa

Array([[-0.3432089 ,  0.        ,  0.        ,  0.22302209],
       [ 0.        , -0.65240585,  0.22302209,  0.        ],
       [ 0.        ,  0.22302209, -0.65240585,  0.        ],
       [ 0.22302209,  0.        ,  0.        , -0.94148065]],      dtype=float64)

In [12]:
vstate = nk.vqs.MCState(sampler, model, n_samples=1008)
optimizer = nk.optimizer.Sgd(learning_rate=0.1)

# Notice the use, again of Stochastic Reconfiguration, which considerably improves the optimisation
gs = nk.driver.VMC(
    ha,
    optimizer,
    variational_state=vstate,
    preconditioner=nk.optimizer.SR(diag_shift=0.1),
)

In [7]:
gs.run(n_iter=300)
ffn_energy = vstate.expect(ha)
# error = abs((ffn_energy.mean - eig_vals[0]) / eig_vals[0])
# print("Optimized energy and relative error: ", ffn_energy, error)

  0%|          | 0/300 [00:00<?, ?it/s]/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/optimizer/qgt/qgt_onthefly.py:139: HolomorphicUndeclaredWarning: 
Defaulting to `holomorphic=False`, but this might lead to increased
computational cost or disabled features. Check if your variational
function is holomorphic, and if so specify `holomorphic=True`as an extra
keyword argument.

To silence this warning, specify the `holomorphic=False/True` keyword
argument.

To numerically check whether your variational function is or not holomorphic
you can use the following snippet:

```python
   vs = nk.vqs.MCState(...)

   nk.utils.is_probably_holomorphic(vs._apply_fun, vs.parameters, vs.samples, vs.model_state)
```

if `nk.utils.is_probably_holomorphic` returns False, then your function is not holomorphic.
If it returns True, it is probably holomorphic.


-------------------------------------------------------
For more detailed informations, visit the following link:
	 https://netket